# Example 6.8 / 1: From Declaration to Generated Constraint

Companion notebook for **6.8 Inspecting a Model** in the AMPL lab Documentation, section "From Declaration to Generated Constraint". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## From Declaration to Generated Constraint

In the Python tab, `ampl` has already read `model.mod`. Entities have `expand()` and `xref()`
methods; `show` and `check` have none and are sent with `ampl.eval()`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;
param rate {PROD} > 0;
param avail >= 0;
var Make {PROD} >= 0;
maximize Output: sum {p in PROD} Make[p];
subject to Time: sum {p in PROD} Make[p] / rate[p] <= avail;
check: avail <= 168;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.param["rate"] = {"bands": 200, "coils": 140}
ampl.param["avail"] = 40

In [5]:
# run.py
ampl.eval("show Time;")
print(ampl.con["Time"].expand())
print(ampl.param["rate"].xref())
ampl.eval("check;")

subject to Time: sum{p in PROD} Make[p]/rate[p] <= avail;
subject to Time:
	0.005*Make['bands'] + 0.00714286*Make['coils'] <= 40;

['Time']


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := bands coils;
param rate := bands 200  coils 140;
param avail := 40;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
show Time;
expand Time;
xref rate;
check;

subject to Time: sum{p in PROD} Make[p]/rate[p] <= avail;
subject to Time:
	0.005*Make['bands'] + 0.00714286*Make['coils'] <= 40;

# 1 entity depends on rate:
Time

